# Fase 4 — Generación de la memoria

Prototipo del Agente Redactor completo (docs/04-generacion.md, tarea 4.4):
recorrer las 5 secciones de la plantilla de la memoria, recuperar del RAG
(Fase 3) el contexto de las 3 secciones que tienen un tipo de documento
propio, generar las 2 secciones de síntesis (Resumen Ejecutivo,
Conclusiones) a partir de las demás ya redactadas, validar cada sección con
el Revisor, y unirlas en el borrador completo.

Reutiliza todo lo ya construido y validado en fases anteriores:
- `src/rag/retriever.py` (Fase 3): recuperación de contexto filtrada por tipo.
- `src/agents/redactor.py` (Fase 2): generación de una sección con Ollama.
- `src/validation/revisor.py`: validación de cifras por código.
- `config/prompts.yaml` / `config/settings.yaml`: plantilla de secciones y
  parámetros del modelo, cargados con `src/config.py`.


In [ ]:
import re
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import RAIZ_PROYECTO, cargar_prompts, cargar_settings
from src.rag.retriever import cargar_vectorstore, recuperar_contexto, contexto_como_texto
from src.rag.enriquecimiento_financiero import extraer_partidas, enriquecer_partidas
from src.rag.loader import cargar_carpeta
from src.agents.redactor import generar_seccion, generar_borrador_completo
from src.validation.revisor import validar_cifras_financieras, validar_cifras_generales


## 4.1 — Cargar el vectorstore y la configuración

**Qué se hace:** se abre la base vectorial ya indexada en la Fase 3
(`data/chroma_db`, 5 chunks de los 3 documentos ficticios) con
`cargar_vectorstore()`, y se cargan `config/settings.yaml` (modelo,
temperature, k) y `config/prompts.yaml` (system prompt + plantilla de
secciones).

**Para qué:** no se reconstruye el RAG aquí — eso ya se validó en el
notebook 02. Este notebook solo lo consume, igual que hará el resto del
pipeline (Streamlit, Fase 6).


In [ ]:
settings = cargar_settings()
prompts = cargar_prompts()

vectorstore = cargar_vectorstore(
    persist_directory=str(RAIZ_PROYECTO / settings["rag"]["persist_directory"]),
    collection_name=settings["rag"]["collection_name"],
    modelo_embeddings=settings["rag"]["modelo_embeddings"],
)

modelo = settings["llm"]["modelo"]
temperature = settings["llm"]["temperature"]
k = settings["rag"]["k"]

for seccion in prompts["secciones"]:
    print(seccion)


## 4.2 — Cargar los documentos originales completos (para el Revisor)

**Qué se hace:** además del contexto que devuelve el RAG (top-k chunks), se
carga el texto **completo** de cada documento original con `cargar_carpeta()`
(Fase 1), agrupado por `tipo`.

**Para qué:** el Revisor debe comparar las cifras del texto generado contra
el documento fuente completo, no solo contra los chunks recuperados — si el
RAG no trajo el chunk con una cifra concreta, pero el modelo la menciona
igualmente (porque coincide con otro fragmento del mismo documento), no
queremos marcarla como inventada por error.


In [ ]:
textos_fuente = {}
for documento in cargar_carpeta(RAIZ_PROYECTO / settings["rag"]["data_raw_directory"]):
    textos_fuente.setdefault(documento.tipo, []).append(documento.texto)
textos_fuente = {tipo: "\n\n".join(partes) for tipo, partes in textos_fuente.items()}

for tipo, texto in textos_fuente.items():
    print(f"[{tipo}] {len(texto)} caracteres")


## 4.3 — Generar las 3 secciones con tipo de documento propio

**Qué se hace:** para "Control Financiero", "Seguimiento de Convenios" y
"Agencia de Colocación" (las únicas con `tipo_documento` en
`config/prompts.yaml`):

1. `recuperar_contexto()` filtrando por `tipo_documento` (nunca sin filtro,
   ver hallazgo de la Fase 3).
2. Si el tipo es `financiero`, enriquecer el contexto con
   `extraer_partidas()` + `enriquecer_partidas()` (hallazgo de la Fase 2:
   el modelo razona mal las relaciones aritméticas si tiene que deducirlas).
3. `generar_seccion()` con el Agente Redactor.
4. Validar con el Revisor: `validar_cifras_generales()` (cualquier tipo) y,
   si es financiero, también `validar_cifras_financieras()`.

**En qué fijarse:** que `incidencias` salga vacío para cada sección. Si no
sale vacío, no continuar sin investigar (mismo criterio que en la Fase 3
con la recuperación).


In [ ]:
resultado_por_titulo = {}

for seccion in prompts["secciones"]:
    tipo = seccion["tipo_documento"]
    if tipo is None:
        continue

    chunks = recuperar_contexto(vectorstore, seccion["pregunta"], tipo, k=k)
    contexto = contexto_como_texto(chunks)

    partidas = None
    if tipo == "financiero":
        partidas = extraer_partidas(contexto)
        contexto = enriquecer_partidas(contexto, partidas)

    texto_generado = generar_seccion(seccion["titulo"], contexto, modelo=modelo, temperature=temperature)

    texto_fuente = textos_fuente.get(tipo, "") + "\n\n" + contexto
    incidencias = validar_cifras_generales(texto_generado, texto_fuente)
    if partidas is not None:
        incidencias += validar_cifras_financieras(texto_generado, partidas)

    resultado_por_titulo[seccion["titulo"]] = {"texto": texto_generado, "incidencias": incidencias}

    print(f"=== {seccion['titulo']} ===")
    print(texto_generado)
    print(f"\nIncidencias: {incidencias}\n")


## Hallazgo: el Revisor general detecta justo el riesgo esperado

Al ejecutar esta celda, "Control Financiero" sale sin incidencias (las cifras
enriquecidas por código coinciden con lo que redacta el modelo). Pero
"Seguimiento de Convenios" y "Agencia de Colocación" sí las generan:

- **Convenios:** el modelo escribe la fecha como "del 1ro de Enero al 30 de
  Abril de 2026" — el documento fuente solo dice "Enero - Abril 2026", sin
  días concretos. `validar_cifras_generales` marca el "1" y el "30" como
  cifras no presentes en la fuente. Es una invención menor (un día de mes
  que no estaba en el dato), pero es exactamente el tipo de precisión falsa
  que el Revisor debe atrapar.
- **Agencia de colocación:** el modelo calcula por su cuenta varios
  porcentajes de crecimiento/tasas de aceptación (ninguno estaba en la
  tabla original, que solo tiene cifras absolutas por año) — el mismo
  riesgo ya documentado en el hallazgo 2.8 de `notebooks/01-test_generacion.ipynb`.
  El Revisor los marca todos para revisión humana.

**Conclusión:** ninguna de las dos incidencias es un "dato mal copiado"
(las cifras que sí vienen del documento son fieles); son cifras que el
modelo añadió por su cuenta. Confirma que `validar_cifras_generales()`
cumple su propósito — no sustituye al criterio humano, pero señala
exactamente los puntos que hay que revisar antes de aceptar el borrador.


## 4.4 — Generar las secciones de síntesis (Resumen Ejecutivo, Conclusiones)

**Decisión de diseño:** "Resumen Ejecutivo" y "Conclusiones" no tienen un
`tipo_documento` propio en `config/prompts.yaml` — no existe un tipo
"resumen" en el RAG. Generarlas con una búsqueda semántica sin filtro de
tipo repetiría el problema ya documentado en la Fase 3 (las cabeceras
institucionales compartidas confunden al embedding entre documentos).

**Solución adoptada:** se generan las últimas, usando como `contexto` el
texto **ya redactado y validado** de las 3 secciones anteriores — no se
vuelve a consultar el RAG. El Revisor valida estas dos secciones contra la
concatenación de los 3 documentos fuente completos (cualquier cifra que
mencionen debería venir de ahí, puesto que solo resumen lo ya dicho).


In [ ]:
contexto_sintesis = "\n\n".join(
    resultado_por_titulo[s["titulo"]]["texto"]
    for s in prompts["secciones"] if s["tipo_documento"] is not None
)
texto_fuente_sintesis = "\n\n".join(textos_fuente.values()) + "\n\n" + contexto_sintesis

for seccion in prompts["secciones"]:
    if seccion["tipo_documento"] is not None:
        continue

    texto_generado = generar_seccion(seccion["titulo"], contexto_sintesis, modelo=modelo, temperature=temperature)
    incidencias = validar_cifras_generales(texto_generado, texto_fuente_sintesis)
    resultado_por_titulo[seccion["titulo"]] = {"texto": texto_generado, "incidencias": incidencias}

    print(f"=== {seccion['titulo']} ===")
    print(texto_generado)
    print(f"\nIncidencias: {incidencias}\n")


## 4.5 — Unir todo en el borrador completo

**Qué se hace:** se concatenan las 5 secciones en el orden de la plantilla
(`config/prompts.yaml`), con su título como encabezado.

**En qué fijarse:** que el borrador se lea de forma coherente de principio
a fin (aunque cada sección se generó de forma independiente) y que ninguna
sección tenga incidencias sin revisar.


In [ ]:
borrador = "\n\n".join(
    f"## {seccion['titulo']}\n\n{resultado_por_titulo[seccion['titulo']]['texto']}"
    for seccion in prompts["secciones"]
)

print(borrador)

print("\n\n--- Resumen de incidencias por sección ---")
for titulo, resultado in resultado_por_titulo.items():
    print(f"{titulo}: {len(resultado['incidencias'])} incidencia(s)")

    if (len(resultado['incidencias']) > 0):
        print(f"Incidencias: {resultado['incidencias']}")


## Código graduado a `src/`

Todo el flujo manual de las celdas anteriores (RAG por sección + enriquecimiento
financiero + generación + Revisor + unión del borrador) ya vive como una única
función en `src/agents/redactor.py`: `generar_borrador_completo(vectorstore)`.
Se prueba a continuación que produce un borrador con la misma estructura
(mismas 5 secciones, en el mismo orden), para poder usarla directamente desde
el resto del pipeline (Streamlit, Fase 6) sin repetir esta lógica.

**Nota:** no se compara el texto exacto entre esta llamada y las celdas
anteriores — `ChatOllama` no es determinista (ver hallazgo 2.6 del notebook
01), así que dos generaciones de la misma sección no producen el mismo texto
palabra por palabra, aunque la fidelidad a las cifras se mantenga.


In [ ]:
resultado_modulo = generar_borrador_completo(vectorstore)

print(resultado_modulo["borrador"])

for titulo, resultado in resultado_modulo["secciones"].items():
    print(f"{titulo}: {len(resultado['incidencias'])} incidencia(s)")

    if (len(resultado['incidencias']) > 0):
        print(f"Incidencias: {resultado['incidencias']}")

# OJO: comparar el orden de las claves del dict "secciones" no sirve, porque
# se rellena primero con las secciones de datos y despues con las de sintesis
# (ver generar_borrador_completo). Lo que importa es el orden real del
# borrador ya unido, que se reconstruye leyendo los encabezados "## ...".
titulos_borrador = re.findall(r"^## (.+)$", resultado_modulo["borrador"], flags=re.MULTILINE)
titulos_esperados = [seccion["titulo"] for seccion in prompts["secciones"]]
print("\n¿El borrador respeta el orden de la plantilla?", titulos_borrador == titulos_esperados)
